In [1]:
%cd /content
!rm -rf worldindex
!git clone https://github.com/RivaanRanawat/worldindex.git worldindex
%cd /content/worldindex

/content
Cloning into 'worldindex'...
remote: Enumerating objects: 166, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 166 (delta 59), reused 145 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (166/166), 1.53 MiB | 3.95 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/worldindex


In [2]:
!curl -sSL https://install.python-poetry.org | python3 -
import os
os.environ["PATH"] += ":/root/.local/bin"
!poetry install --with video

Retrieving Poetry metadata

# Welcome to Poetry!

This will download and install the latest version of Poetry,
a dependency and package manager for Python.

It will add the `poetry` command to Poetry's bin directory, located at:

/root/.local/bin

You can uninstall at any time by executing this script with the --uninstall option,
and these changes will be reverted.

Installing Poetry (2.3.2)
Installing Poetry (2.3.2): Creating environment
Installing Poetry (2.3.2): Installing Poetry
Installing Poetry (2.3.2): Creating script
Installing Poetry (2.3.2): Done

Poetry (2.3.2) is installed now. Great!

To get started you need Poetry's bin directory (/root/.local/bin) in your `PATH`
environment variable.

Add `export PATH="/root/.local/bin:$PATH"` to your shell configuration file.

Alternatively, you can call Poetry explicitly with `/root/.local/bin/poetry`.

You can test that everything is set up by executing:

`poetry --version`

Creating virtualenv worldindex in /content/worldindex/.venv


In [8]:
%%bash

cat > colab_extract_aloha.py <<'PY'
from pathlib import Path

from extraction import ExtractionConfig, run_extraction
from ingestion.config import DatasetConfig


def main() -> None:
    config = ExtractionConfig(
        dataset_configs=[DatasetConfig.from_yaml(Path("config/datasets/aloha.yaml"))],
        model_id="facebook/vjepa2-vith-fpc64-256",
        device="cuda",
        batch_size=1,
        queue_depth=2,
        flush_size=10,
        start_method="spawn",
        clip_former_factory="tests.extraction.spawn_fakes:build_limited_real_clip_former",
        output_dir=Path("artifacts/extraction/gpu/aloha_smoke"),
        checkpoint_db=Path("state/gpu/extraction_aloha_smoke.sqlite3"),
    )

    last_clip_index = run_extraction(config)
    print({"last_clip_index": last_clip_index, "output_dir": str(config.output_dir)})


if __name__ == "__main__":
    main()
PY


In [17]:
!WORLDINDEX_REAL_EXTRACTION_CLIP_LIMIT=20 poetry run python colab_extract_aloha.py

2026-03-02 07:52:41 [info     ] extraction_progress            component=single_node_extraction last_clip_index=13 processed_clips=14 resume_from=14
2026-03-02 07:52:47 [info     ] extraction_progress            component=single_node_extraction last_clip_index=13 processed_clips=14 resume_from=14
`torch_dtype` is deprecated! Use `dtype` instead!
2026-03-02 07:52:53 [info     ] extraction_progress            component=single_node_extraction last_clip_index=13 processed_clips=14 resume_from=14
2026-03-02 07:52:58 [debug    ] prepared_episode               episode_id=lerobot/aloha_sim_insertion_scripted_image_0 repo_id=lerobot/aloha_sim_insertion_scripted_image source_frames=400 subsampled_frames=32
2026-03-02 07:52:59 [info     ] extraction_progress            component=single_node_extraction last_clip_index=13 processed_clips=14 resume_from=14
2026-03-02 07:53:00 [debug    ] prepared_episode               episode_id=lerobot/aloha_sim_insertion_scripted_image_1 repo_id=lerobot/aloha_sim_

In [18]:
%%bash
cd /content/worldindex
poetry run python - <<'PY'
from pathlib import Path
import numpy as np

token_paths = sorted(Path("artifacts/extraction/gpu/aloha_smoke").glob("tokens_*.npy"))
print([path.name for path in token_paths])

total_clips = 0
for path in token_paths:
    arr = np.load(path, mmap_mode="r")
    total_clips += int(arr.shape[0])

print({"token_batches": len(token_paths), "total_clips": total_clips})
PY


['tokens_00000000_00000009.npy', 'tokens_00000010_00000013.npy']
{'token_batches': 2, 'total_clips': 14}


In [26]:
%%bash

mkdir -p config/local

cat > config/local/compress_aloha_smoke.yaml <<'YAML'
raw_dir: artifacts/extraction/gpu/aloha_smoke
output_dir: artifacts/serving/aloha_smoke
checkpoint_db: state/compression_aloha_smoke.sqlite3
sample_size: 16384
pca_dim: 128
n_centroids: 128
n_bits: 2
random_seed: 0
YAML


In [29]:
!cat config/local/compress_aloha_smoke.yaml

raw_dir: artifacts/extraction/gpu/aloha_smoke
output_dir: artifacts/serving/aloha_smoke
checkpoint_db: state/compression_aloha_smoke.sqlite3
sample_size: 16384
pca_dim: 128
n_centroids: 128
n_bits: 2
random_seed: 0


In [30]:
!poetry run python scripts/compress.py config/local/compress_aloha_smoke.yaml

2026-03-02 08:16:17 [info     ] pca_variance_below_target      component=token_compressor explained_variance=0.8098633885383606 fallback_pca_dim=256 requested_pca_dim=128
2026-03-02 08:16:23 [info     ] training_complete              component=token_compressor explained_variance=0.9028439521789551 input_dim=1280 n_centroids=128 pca_dim=256
2026-03-02 08:16:23 [info     ] trained_compressor             component=compression_pipeline compressor_dir=artifacts/serving/aloha_smoke/compression_model
2026-03-02 08:16:24 [info     ] compressed_shard_written       clip_count=10 component=compression_pipeline shard_id=0 shard_path=artifacts/serving/aloha_smoke/shards/shard_00000000.widx
2026-03-02 08:16:25 [info     ] compressed_shard_written       clip_count=4 component=compression_pipeline shard_id=1 shard_path=artifacts/serving/aloha_smoke/shards/shard_00000001.widx
2026-03-02 08:16:25 [info     ] faiss_index_built              component=index_builder dim=256 index_path=artifacts/serving/aloh

In [32]:
%%bash
ls artifacts/serving/aloha_smoke
ls artifacts/serving/aloha_smoke/shards


clips.parquet
coarse_hnsw.faiss
compression_model
metadata
shards
shard_00000000.widx
shard_00000001.widx


In [33]:
%%bash
cat > config/local/serve_aloha_smoke.yaml <<'YAML'
host: 127.0.0.1
port: 8000
data_dir: artifacts/serving/aloha_smoke
model_id: facebook/vjepa2-vith-fpc64-256
device: cuda
max_concurrent_queries: 1
YAML

In [ ]:
%%bash
!poetry run python scripts/serve.py config/local/serve_aloha_smoke.yaml

Traceback (most recent call last):
  File "/content/worldindex/scripts/serve.py", line 8, in <module>
    from api.config import ServingConfig
ModuleNotFoundError: No module named 'api'
